In [1]:
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import *

2026-07-13 12:12:30.181454: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-13 12:12:30.199864: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-13 12:12:30.199882: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-13 12:12:30.200426: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-13 12:12:30.203581: I tensorflow/core/platform/cpu_feature_guar

In [2]:
with h5py.File('processed_physics_data.h5', 'r') as f:
    X_train = np.expand_dims(f['X_train'][:], -1)
    y_train = f['y_train'][:]
    X_val = np.expand_dims(f['X_val'][:], -1)
    y_val = f['y_val'][:]

In [3]:
inputs = Input(shape=(24, 36, 1))
x = Conv2D(64, (3, 3), padding='same')(inputs)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D((2, 2))(x)

2026-07-13 12:12:31.465768: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-07-13 12:12:31.484819: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-07-13 12:12:31.484988: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [4]:
x = Conv2D(128, (3, 3), padding='same')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D((2, 2))(x)

In [5]:
sp_att = Conv2D(1, (3, 3), padding='same', activation='sigmoid')(x)
x = Multiply()([x, sp_att])
x = MaxPooling2D((2, 2))(x)

In [6]:
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

In [7]:
outputs = Dense(1, activation='sigmoid')(x)
focal_model = Model(inputs, outputs)
focal_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 24, 36, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 24, 36,    │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 24, 36,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 24, 36,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 12, 18,    │          0 │ activation[0][0]  │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 12, 18,    │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 12, 18,    │        512 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 12, 18,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 6, 9, 128) │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 6, 9, 1)   │      1,153 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 6, 9, 128) │          0 │ max_pooling2d_1[… │
│                     │                   │            │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 3, 4, 128) │          0 │ multiply[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    393,472 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        257 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 471,170 (1.80 MB)

 Trainable params: 470,274 (1.79 MB)

 Non-trainable params: 896 (3.50 KB)

In [8]:
focal_loss = tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.9)
focal_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), 
                    loss=focal_loss, 
                    metrics=[tf.keras.metrics.AUC(name='auc')])

In [9]:
callbacks = [
    ModelCheckpoint('focal_bhabha_model.h5', monitor='val_auc', save_best_only=True, mode='max'),
    EarlyStopping(monitor='val_auc', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=4, min_lr=1e-6)
]

In [10]:
history = focal_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200, batch_size=32,
    callbacks=callbacks, verbose=1
)

Epoch 1/200


2026-07-13 12:12:33.152556: I external/local_xla/xla/service/service.cc:168] XLA service 0x5957954d86b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-07-13 12:12:33.152584: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GT 1030, Compute Capability 6.1
2026-07-13 12:12:33.183796: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-07-13 12:12:33.360889: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902


   6/2744 ━━━━━━━━━━━━━━━━━━━━ 1:07 25ms/step - auc: 0.4217 - loss: 0.8280

I0000 00:00:1783912358.370148 3406890 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2744/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - auc: 0.6890 - loss: 0.1552

...

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 72s 26ms/step - auc: 0.8635 - loss: 0.0595 - val_auc: 0.8701 - val_loss: 0.0588 - learning_rate: 0.0010
Epoch 16/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8659 - loss: 0.0593 - val_auc: 0.8693 - val_loss: 0.0588 - learning_rate: 0.0010
Epoch 17/200
2742/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - auc: 0.8661 - loss: 0.0594

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8643 - loss: 0.0594 - val_auc: 0.8702 - val_loss: 0.0585 - learning_rate: 0.0010
Epoch 18/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8653 - loss: 0.0592 - val_auc: 0.8693 - val_loss: 0.0588 - learning_rate: 0.0010
Epoch 19/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 72s 26ms/step - auc: 0.8666 - loss: 0.0590 - val_auc: 0.8693 - val_loss: 0.0587 - learning_rate: 0.0010
Epoch 20/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - auc: 0.8687 - loss: 0.0587

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8710 - loss: 0.0583 - val_auc: 0.8713 - val_loss: 0.0582 - learning_rate: 5.0000e-04
Epoch 21/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - auc: 0.8716 - loss: 0.0576

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8716 - loss: 0.0581 - val_auc: 0.8715 - val_loss: 0.0581 - learning_rate: 5.0000e-04
Epoch 22/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8725 - loss: 0.0579 - val_auc: 0.8707 - val_loss: 0.0583 - learning_rate: 5.0000e-04
Epoch 23/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 72s 26ms/step - auc: 0.8731 - loss: 0.0578 - val_auc: 0.8714 - val_loss: 0.0582 - learning_rate: 5.0000e-04
Epoch 24/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8747 - loss: 0.0574 - val_auc: 0.8695 - val_loss: 0.0585 - learning_rate: 5.0000e-04
Epoch 25/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8743 - loss: 0.0575 - val_auc: 0.8706 - val_loss: 0.0584 - learning_rate: 5.0000e-04
Epoch 26/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8775 - loss: 0.0569 - val_auc: 0.8714 - val_loss: 0.0582 - learning_rate: 2.5000e-04
Epoch 27/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8782 - loss: 0.0567 - val_auc: 0.8

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8776 - loss: 0.0567 - val_auc: 0.8716 - val_loss: 0.0583 - learning_rate: 2.5000e-04
Epoch 29/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 72s 26ms/step - auc: 0.8781 - loss: 0.0567 - val_auc: 0.8707 - val_loss: 0.0585 - learning_rate: 2.5000e-04
Epoch 30/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 72s 26ms/step - auc: 0.8800 - loss: 0.0562 - val_auc: 0.8704 - val_loss: 0.0584 - learning_rate: 1.2500e-04
Epoch 31/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8814 - loss: 0.0561 - val_auc: 0.8712 - val_loss: 0.0585 - learning_rate: 1.2500e-04
Epoch 32/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8826 - loss: 0.0558 - val_auc: 0.8706 - val_loss: 0.0585 - learning_rate: 1.2500e-04
Epoch 33/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8813 - loss: 0.0560 - val_auc: 0.8712 - val_loss: 0.0585 - learning_rate: 1.2500e-04
Epoch 34/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 71s 26ms/step - auc: 0.8829 - loss: 0.0557 - val_auc: 0.8